In [1]:
import pandas as pd

from superlinked import framework as sl

pd.set_option("display.max_colwidth", 200)

In [2]:
data_in_path: str = "./data/processed/product.csv"

In [27]:
prod_df = pd.read_csv(data_in_path).drop_duplicates(subset=["productId"])
prod_df["productId"] = prod_df["productId"].astype(str)

In [4]:
prod_df["full_text_data"] =  prod_df.apply(
    lambda row: ', '.join(f'{col}: {row[col]}' for col in prod_df.columns if not col == "productId"), axis=1)

In [5]:
class Product(sl.Schema):
    productId: sl.IdField
    full_text_data: sl.String

product = Product()

In [6]:
text_space = sl.TextSimilaritySpace(
    product.full_text_data, 
    model="NYTK/sentence-transformers-experimental-hubert-hungarian"
)

In [7]:
index = sl.Index([text_space])

In [8]:
source: sl.InMemorySource = sl.InMemorySource(product, parser=sl.DataFrameParser(product))
executor = sl.InMemoryExecutor(sources=[source], indices=[index])
app = executor.run()

In [37]:
query = sl.Query(index).find(product).similar(text_space, sl.Param("query_text")).select_all().limit(5)

In [10]:
import time

st = time.time()
source.put(prod_df)
end = time.time()

In [14]:
print(f"Time took: {(end - st) / 60}  minutes")

Time took: 4.141536545753479  minutes


In [23]:
result = app.query(query, query_text="mosószer")

In [38]:
def show_result(result: "Result", product_df: pd.DataFrame = prod_df) -> pd.DataFrame:
    return sl.PandasConverter.to_pandas(result).merge(product_df, left_on="id", right_on="productId")

In [39]:
show_result(app.query(query, query_text="mosószer"))

,full_text_data,id,similarity_score,productId,description,unified_ingredients,energyKJ,energyKCal,fats,saturatedFats,carbohydrates,sugars,protein,salt,fiber,allergen_contain,allergen_possibly_contain,price
0,"description: A mosószóda (nátrium karbonát, sziksó) **biológiailag lebomló, vízlágyító,\náztató és jó zsíroldó tisztítószer**. A mosószódát használhatod mosáshoz\nönállóan a ""hagyományos"" mosópor ...",87124,0.790396,87124,"A mosószóda (nátrium karbonát, sziksó) **biológiailag lebomló, vízlágyító,\náztató és jó zsíroldó tisztítószer**. A mosószódát használhatod mosáshoz\nönállóan a ""hagyományos"" mosópor helyett vagy ...",95-99% Nátrium-karbonát,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],1599.0
1,"description: **Aktív oxigénforrás** , melegvízben való szétbomlása során oxigén, víz és\nnátrium-karbonát (mosószóda) keletkezik. A felszabaduló aktív oxigén erős\ntisztító, fehérítő, folteltávolí...",87123,0.772668,87123,"**Aktív oxigénforrás** , melegvízben való szétbomlása során oxigén, víz és\nnátrium-karbonát (mosószóda) keletkezik. A felszabaduló aktív oxigén erős\ntisztító, fehérítő, folteltávolító, szagtalan...","nátrium-perkarbonát, Aktivátor",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],3439.0
2,"description: **Apart preBIOtic folyékony szappan utántöltő, Iris & Jasmine**\n\nAz APART Floral Care folyékony szappan finoman tisztítja és ápolja a bőrt.\nPrebiotikummal gazdagított formula.\n\n*...",101249,0.770692,101249,"**Apart preBIOtic folyékony szappan utántöltő, Iris & Jasmine**\n\nAz APART Floral Care folyékony szappan finoman tisztítja és ápolja a bőrt.\nPrebiotikummal gazdagított formula.\n\n**Használat**\...","Aqua, Sodium Laureth Sulfate, Glycerin, Sodium Chloride, Cocamidopropyl Betaine, Lauryl Hydroxysultaine, Acrylates Copolymer, Inulin, Coco-Caprylate, Lauryl Glucoside, Polyglyceryl-2 Dipolyhydroxy...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],1099.0
3,"description: **Cif Cleanboost univerzális vízkőoldó**\n\nAlighogy befejezted a fürdőszoba takarítását, az élet már megy is tovább, és a\nvízcsepp foltok és szappanlerakódások nyomban megjelennek ú...",9733,0.766209,9733,"**Cif Cleanboost univerzális vízkőoldó**\n\nAlighogy befejezted a fürdőszoba takarítását, az élet már megy is tovább, és a\nvízcsepp foltok és szappanlerakódások nyomban megjelennek újra. Ezért\nf...","limonene, benzisothiazolinone, nem ionos felületaktív anyagok, illatanyag",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],1799.0
4,"description: A lanolinos Baba szappan a Baba márka legrégebbi terméke, már több, mint száz\néve nyújt az egész család számára gyengéd ápolást. A mindennapos használatra\nkifejlesztett, bőrápoló ös...",92838,0.764235,92838,"A lanolinos Baba szappan a Baba márka legrégebbi terméke, már több, mint száz\néve nyújt az egész család számára gyengéd ápolást. A mindennapos használatra\nkifejlesztett, bőrápoló összetevőket ta...","Sodium Tallowate (A)/ Sodium Palmate (B)*, Aqua, Sodium Palm Kernelate, Glycerin, Lanolin, Palm Kernel Acid, Parfum, Sodium Chloride, Tetrasodium EDTA, Tetrasodium Etidronate, Geraniol, Hexyl Cinn...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],279.0


In [40]:
show_result(app.query(query, query_text="alacsony kalóriatartalmú snack"))

,full_text_data,id,similarity_score,productId,description,unified_ingredients,energyKJ,energyKCal,fats,saturatedFats,carbohydrates,sugars,protein,salt,fiber,allergen_contain,allergen_possibly_contain,price
0,"description: Ázsiában őshonos zöldség, a nyers kínai kelnek magas folsav- és C-vitamin\ntartalma van, emellett szénhidrát-tartalma csekély, fehérjetartalma szintén,\nkönnyen emészthető, kedvező am...",70135,0.639713,70135,"Ázsiában őshonos zöldség, a nyers kínai kelnek magas folsav- és C-vitamin\ntartalma van, emellett szénhidrát-tartalma csekély, fehérjetartalma szintén,\nkönnyen emészthető, kedvező aminosav-összet...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1349.0
1,"description: **Pörkölt héjas földimogyoró**\n\nA földimogyoró magas zsírtartalma ellenére gazdag A-, E- és C-vitaminban,\nkalciumban, nátriumban és rostban, vagyis azokban a tápanyagokban, amelye...",82288,0.639200,82288,"**Pörkölt héjas földimogyoró**\n\nA földimogyoró magas zsírtartalma ellenére gazdag A-, E- és C-vitaminban,\nkalciumban, nátriumban és rostban, vagyis azokban a tápanyagokban, amelyeket a\nlegrit...",Földimogyoró,2491.0,602.0,50.0,5.5,10.0,4.7,25.0,0.03,8.7,['Földimogyoró'],[],669.0
2,"description: A cékla rengeteg vitamint tartalmaz, ezért azokban az időszakokban amikor\ngyengének és fáradtnak érezzük magunkat kifejezetten javasolt a fogyasztása.\nMivel nem rendelkezik kifejeze...",24127,0.628893,24127,"A cékla rengeteg vitamint tartalmaz, ezért azokban az időszakokban amikor\ngyengének és fáradtnak érezzük magunkat kifejezetten javasolt a fogyasztása.\nMivel nem rendelkezik kifejezetten jellegze...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.0
3,"description: **Földimogyoró wasabis**\n\nA földimogyoró magas zsírtartalma ellenére gazdag A-, E- és C-vitaminban,\nkalciumban, nátriumban és rostban, vagyis azokban a tápanyagokban, amelyeket a\n...",82657,0.626469,82657,"**Földimogyoró wasabis**\n\nA földimogyoró magas zsírtartalma ellenére gazdag A-, E- és C-vitaminban,\nkalciumban, nátriumban és rostban, vagyis azokban a tápanyagokban, amelyeket a\nlegritkábban ...","földimogyoróbél, Maltodextrin, Cukor, spenótpor, Aroma, Só, Élesztő kivonat, savszabályzó, hidrolizált növényi fehérjék, Növényi olaj, Csomósodásgátló anyag",2574.0,622.0,51.2,5.0,11.1,6.6,25.1,1.00,6.0,['Földimogyoró'],"['Diófélék', 'Szezámmag']",679.0
4,"description: **Éden Prémium Easy Pasta Vöröslencse tészta orsó**\n\nA **hüvelyesek** jó forrásai a fehérjéknek, a telítetlen zsírsavaknak, a\nrostoknak, és magas vitamin-, és ásványianyag-tartalom...",61598,0.612871,61598,"**Éden Prémium Easy Pasta Vöröslencse tészta orsó**\n\nA **hüvelyesek** jó forrásai a fehérjéknek, a telítetlen zsírsavaknak, a\nrostoknak, és magas vitamin-, és ásványianyag-tartalommal rendelkez...","Vöröslencse liszt, víz, útifűmaghéjliszt",1499.0,358.0,1.4,0.4,53.0,5.0,21.0,0.01,18.0,[],['Diófélék'],999.0
